In [ ]:
import os
import json
from collections import Counter, defaultdict
from pathlib import Path

PATH_RAW_REVIEWS = os.path.join('..', 'datasets', 'raw', 'Cell_Phones_and_Accessories.jsonl')
PATH_RAW_ITEMS = os.path.join('..', 'datasets', 'raw', 'meta_Cell_Phones_and_Accessories.jsonl')
PATH_OUT_DIR = os.path.join('..', 'datasets', 'processed')
PATH_GT_DIR = os.path.join('..', 'datasets', 'ground_truth')

os.makedirs(PATH_OUT_DIR, exist_ok=True)
os.makedirs(PATH_GT_DIR, exist_ok=True)

MIN_REVIEWS_PER_USER = 5
MAX_USERS = 2000


def count_user_reviews(review_path):
    user_counts = Counter()
    with open(review_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                obj = json.loads(line)
                user_id = obj.get('user_id')
                if user_id:
                    user_counts[user_id] += 1
            except Exception:
                continue
    return user_counts


def collect_interactions(review_path, target_users):
    user_interactions = defaultdict(list)
    with open(review_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                obj = json.loads(line)
                user_id = obj.get('user_id')
                if user_id not in target_users:
                    continue

                item_id = obj.get('parent_asin') or obj.get('asin')
                if not item_id:
                    continue

                user_interactions[user_id].append({
                    'user_id': user_id,
                    'item_id': item_id,
                    'rating': obj.get('rating'),
                    'timestamp': obj.get('timestamp', 0)
                })
            except Exception:
                continue
    return user_interactions


def collect_item_metadata(meta_path, target_items):
    item_metadata = []
    with open(meta_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                obj = json.loads(line)
                item_id = obj.get('parent_asin') or obj.get('asin')
                if item_id in target_items:
                    item_metadata.append({
                        'item_id': item_id,
                        'title': obj.get('title', ''),
                        'categories': obj.get('categories', []),
                        'brand': obj.get('brand', '')
                    })
            except Exception:
                continue
    return item_metadata


print('Đang đếm số review mỗi user...')
user_counts = count_user_reviews(PATH_RAW_REVIEWS)
active_users = {user_id for user_id, count in user_counts.items() if count >= MIN_REVIEWS_PER_USER}
print(f'Tổng user có review: {len(user_counts)}')
print(f'User hoạt động >= {MIN_REVIEWS_PER_USER} review: {len(active_users)}')

selected_users = set(
    user_id for user_id, _ in sorted(user_counts.items(), key=lambda x: x[1], reverse=True)[:MAX_USERS]
    if user_id in active_users
)
print(f'Selected top {len(selected_users)} users cho dataset dense')

print('Đang thu thập interaction cho các user được chọn...')
user_interactions = collect_interactions(PATH_RAW_REVIEWS, selected_users)

train_interactions = []
ground_truth = {}
target_items = set()

for user_id, interactions in user_interactions.items():
    interactions = sorted(interactions, key=lambda x: x['timestamp'])
    if len(interactions) <= 1:
        continue

    split_idx = int(len(interactions) * 0.8)
    if split_idx == 0:
        split_idx = 1
    if split_idx >= len(interactions):
        split_idx = max(1, len(interactions) - 1)

    train_list = interactions[:split_idx]
    test_list = interactions[split_idx:]

    train_interactions.extend(train_list)
    ground_truth[user_id] = [item['item_id'] for item in test_list]
    for item in train_list + test_list:
        target_items.add(item['item_id'])

print(f'Train interactions: {len(train_interactions)}')
print(f'Ground truth users: {len(ground_truth)}')
print(f'Unique items: {len(target_items)}')

print('Đang thu thập metadata cho các item...')
item_metadata = collect_item_metadata(PATH_RAW_ITEMS, target_items)
print(f'Found metadata for {len(item_metadata)} items')

train_data = {
    'interactions': train_interactions,
    'items': item_metadata
}

with open(os.path.join(PATH_OUT_DIR, 'train_data.json'), 'w', encoding='utf-8') as f:
    json.dump(train_data, f, ensure_ascii=False, indent=2)

with open(os.path.join(PATH_GT_DIR, 'ground_truth.json'), 'w', encoding='utf-8') as f:
    json.dump(ground_truth, f, ensure_ascii=False, indent=2)

print('Đã lưu train_data.json và ground_truth.json')
print('Done!')

Đang đếm số review mỗi user...
Tổng user có review: 11598197
User hoạt động >= 5 review: 608090
Selected top 2000 users cho dataset dense
Đang thu thập interaction cho các user được chọn...
Train interactions: 80442
Ground truth users: 2000
Unique items: 57118
Đang thu thập metadata cho các item...
Found metadata for 57118 items
Đã lưu train_data.json và ground_truth.json
Done!
